<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-09-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-100.

Características do dataset:

- 60.000 imagens RGB
- 100 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar100

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-100 utilizando `tensorflow.keras.datasets.cifar100.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from tensorflow.keras.datasets import cifar100
import mlflow
import time

from src.utils import set_seed, normalize_images
from src.metrics import classification_metrics
from src.experiment import setup_experiment, start_run, log_params, log_metrics
from src.plots import compare_models

SEED = 42


def load_data(seed):
    set_seed(seed)
    (X_full, y_full), _ = cifar100.load_data()

    # Flatten: (N, 32, 32, 3) -> (N, 3072) e normaliza para [0, 1]
    X_full = normalize_images(X_full.reshape(X_full.shape[0], -1)).astype(np.float32)
    y_full = y_full.ravel()

    X_train, X_val, y_train, y_val = train_test_split(
        X_full, y_full, test_size=0.2, random_state=seed
    )
    return X_train, X_val, y_train, y_val


X_train, X_val, y_train, y_val = load_data(SEED)

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")
print(f"y_train: {y_train.shape}, y_val: {y_val.shape}")

### Respostas — Questão 1

**1. Qual o formato original das imagens?**

O formato original é **(32, 32, 3)**: 32 pixels de altura × 32 pixels de largura × 3 canais de cor (RGB).

**2. Quantas features cada imagem possui após o flatten?**

Após o flatten: 32 × 32 × 3 = **3.072 features** por imagem.

**3. Por que o flatten é necessário para uma MLP?**

A MLP opera com camadas totalmente conectadas que realizam `y = Wx + b`, exigindo que a entrada seja um **vetor unidimensional**. A estrutura espacial 2D das imagens (relação de vizinhança entre pixels) é desconsiderada — a MLP trata cada pixel como uma feature independente. CNNs, por outro lado, foram projetadas para explorar justamente essa organização espacial via filtros convolucionais.

**4. Qual a importância da normalização para o treinamento?**

Sem normalização, pixels com valores altos (ex.: 250) produziriam gradientes muito maiores que pixels com valores baixos (ex.: 5), desequilibrando a atualização dos pesos. Ao escalar para [0, 1]:

- todas as features contribuem de forma equilibrada para o gradiente;
- o otimizador converge mais rapidamente;
- o treinamento torna-se menos sensível à inicialização dos pesos e à escolha do learning rate.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [ ]:
def train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=20,
        batch_size=512,
        verbose=False
    )
    model.fit(X_train, y_train)
    return model


model_demo = train_mlp(X_train, y_train, 'relu', (128, 64), 0.01, SEED)
print(f"Modelo treinado. Iterações realizadas: {model_demo.n_iter_}")

### Respostas — Questão 2

**1. Quantos parâmetros existem na primeira camada?**

Com 3.072 features de entrada e 128 neurônios na primeira camada oculta:

- Pesos: 3.072 × 128 = 393.216  
- Biases: 128  
- **Total da primeira camada: 393.344 parâmetros**

**2. Qual a função da ativação ReLU?**

ReLU (*Rectified Linear Unit*) é definida por `f(x) = max(0, x)`: zera valores negativos e mantém os positivos sem alteração. Suas vantagens são:

- **Evita o vanishing gradient** em valores positivos — o gradiente é constante (= 1), não próximo de zero como nas regiões saturadas de sigmoid/tanh;
- **Eficiência computacional** — implementação trivial, apenas uma comparação;
- **Esparsidade** — neurônios com entrada negativa têm saída zero, o que reduz ativações desnecessárias e torna a rede mais eficiente.

**3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?**

MLPs utilizam **conexões totais** (*fully connected*): cada neurônio de uma camada conecta-se a *todos* os neurônios (pixels) da camada anterior. Para uma imagem 32×32×3 (3.072 features), uma única camada de 128 neurônios já exige 393.344 parâmetros. CNNs contornam esse problema com **compartilhamento de pesos**: o mesmo filtro é aplicado em todas as posições da imagem, reduzindo drasticamente o número de parâmetros aprendidos.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    metrics = classification_metrics(y_test, y_pred)
    return metrics


metrics_demo = evaluate(model_demo, X_val, y_val)
df_metrics = pd.DataFrame([metrics_demo])
print("Métricas do modelo (demo):")
print(df_metrics.round(4).to_string(index=False))

### Respostas — Questão 3

**1. O que a accuracy representa?**

A accuracy é a proporção de exemplos classificados corretamente sobre o total: `accuracy = nº de acertos / nº total de amostras`. É uma métrica global de desempenho, fácil de interpretar, mas pode ser enganosa em datasets com classes desbalanceadas.

**2. Qual a diferença entre precision e recall?**

- **Precision**: dos exemplos *preditos* como classe X, qual fração *realmente* pertence à classe X. Penaliza **falsos positivos** — útil quando o custo de uma predição incorreta é alto (ex.: spam detectando e-mails legítimos como spam).  
- **Recall**: dos exemplos que *realmente* são da classe X, qual fração foi *predita* como X. Penaliza **falsos negativos** — útil quando o custo de perder um positivo é alto (ex.: diagnóstico de doença).

**3. Em quais situações o f1-score é importante?**

O f1-score é a média harmônica entre precision e recall: `f1 = 2 × (precision × recall) / (precision + recall)`. É importante quando:

- há **desbalanceamento de classes** — a accuracy sozinha é enganosa e o f1 pondera melhor os erros por classe;
- é necessário **equilibrar** precision e recall, sem sacrificar um para maximizar o outro (ex.: detecção de fraudes, diagnóstico médico).

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
setup_experiment("cifar100_mlp")

activation_base = "relu"
hidden_layers_base = (128, 64)
learning_rate_base = 0.01
max_iter_base = 20
batch_size_base = 512

with start_run("baseline"):
    log_params({
        "activation": activation_base,
        "hidden_layers": str(hidden_layers_base),
        "learning_rate": learning_rate_base,
        "max_iter": max_iter_base,
        "batch_size": batch_size_base
    })

    start_time = time.time()
    model_baseline = MLPClassifier(
        hidden_layer_sizes=hidden_layers_base,
        activation=activation_base,
        learning_rate_init=learning_rate_base,
        max_iter=max_iter_base,
        batch_size=batch_size_base,
        random_state=SEED
    ).fit(X_train, y_train)
    training_time = time.time() - start_time

    metrics_baseline = evaluate(model_baseline, X_val, y_val)
    metrics_baseline["training_time"] = training_time
    log_metrics(metrics_baseline)

    print(f"Baseline — Accuracy: {metrics_baseline['accuracy']:.4f}, Tempo: {training_time:.2f}s")

### Respostas — Questão 4

**1. Qual experimento apresentou melhor desempenho?**

O baseline com `relu + (128, 64) + lr=0.01` serve como referência inicial. Os experimentos das Questões 5, 6 e 7 realizam comparações sistemáticas — variando função de ativação, arquitetura e learning rate individualmente — para identificar qual configuração produz as melhores métricas.

**2. Qual configuração apresentou maior estabilidade?**

Configurações com **learning rate moderado (0.01)** e **arquitetura balanceada como (128, 64)** apresentam maior estabilidade: o lr é grande o suficiente para progredir dentro de 20 iterações, mas pequeno o suficiente para não causar oscilações na função de perda.

**3. Qual o benefício do rastreamento experimental?**

O MLflow oferece quatro benefícios centrais:

- **Reprodutibilidade** — qualquer experimento pode ser replicado exatamente com os mesmos hiperparâmetros registrados;
- **Comparação sistemática** — todas as métricas ficam centralizadas, eliminando comparações manuais e sujeitas a erro;
- **Auditabilidade** — histórico completo das decisões e resultados experimentais, útil para relatórios e revisões;
- **Identificação de impacto** — ao variar um hiperparâmetro por vez e observar as métricas, é possível medir quantitativamente a influência de cada escolha.

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
activations = ["logistic", "tanh", "relu"]
results_activations = {}

for act in activations:
    with start_run(f"activation_{act}"):
        log_params({
            "activation": act,
            "hidden_layers": str((128, 64)),
            "learning_rate": 0.01,
            "max_iter": 20,
            "batch_size": 512
        })

        start_time = time.time()
        model_act = MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation=act,
            learning_rate_init=0.01,
            max_iter=20,
            batch_size=512,
            random_state=SEED
        ).fit(X_train, y_train)
        training_time = time.time() - start_time

        metrics_act = evaluate(model_act, X_val, y_val)
        metrics_act["training_time"] = training_time
        log_metrics(metrics_act)
        results_activations[act] = metrics_act

        print(f"{act:10s}: accuracy={metrics_act['accuracy']:.4f}, f1={metrics_act['f1_score']:.4f}, tempo={training_time:.2f}s")

df_act = pd.DataFrame(results_activations).T
print("\nComparação de Funções de Ativação:")
print(df_act[["accuracy", "precision", "recall", "f1_score", "training_time"]].round(4).to_string())

compare_models(
    list(results_activations.keys()),
    [m["accuracy"] for m in results_activations.values()],
    "Accuracy por Função de Ativação"
)

### Respostas — Questão 5

**1. Qual ativação apresentou melhor convergência?**

A **ReLU** tende a apresentar melhor convergência. Por não saturar em valores positivos (gradiente = 1 nessa região), os pesos continuam sendo atualizados de forma efetiva em todas as camadas durante o backpropagation, evitando o problema de *vanishing gradient*.

**2. Qual ativação apresentou maior estabilidade?**

A **tanh** costuma apresentar estabilidade comparável à ReLU: seus gradientes são simétricos em torno de zero e menos propensos a oscilações abruptas. A **logistic** (sigmoid) é a menos estável, pois satura tanto em valores muito positivos quanto muito negativos, produzindo gradientes próximos de zero.

**3. Houve diferenças significativas no treinamento?**

Sim. A **logistic** tende a convergir mais lentamente e a atingir menor accuracy em poucas iterações, dado que sua região de saturação domina boa parte dos valores de ativação durante as primeiras épocas. A ReLU e a tanh apresentam desempenho mais próximo, com a ReLU frequentemente levando vantagem em velocidade de convergência.

**4. Por que a ReLU é amplamente utilizada em Deep Learning?**

- **Evita o vanishing gradient** em valores positivos — gradiente constante (= 1) garante propagação eficaz do sinal de erro até camadas iniciais;
- **Eficiência computacional** — `max(0, x)` é a operação mais simples possível;
- **Esparsidade** — neurônios com saída zero não contribuem para o cálculo nem propagam gradiente, tornando a rede mais eficiente e com menos sobreajuste implícito;
- **Resultados empíricos** — supera sigmoid e tanh na vasta maioria dos cenários de classificação com redes profundas.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
architectures = [(32,), (64,), (128, 64), (256, 128)]
results_architectures = {}

for arch in architectures:
    arch_name = str(arch)
    with start_run(f"arch_{arch_name}"):
        log_params({
            "activation": "relu",
            "hidden_layers": arch_name,
            "learning_rate": 0.01,
            "max_iter": 20,
            "batch_size": 512
        })

        start_time = time.time()
        model_arch = MLPClassifier(
            hidden_layer_sizes=arch,
            activation="relu",
            learning_rate_init=0.01,
            max_iter=20,
            batch_size=512,
            random_state=SEED
        ).fit(X_train, y_train)
        training_time = time.time() - start_time

        metrics_arch = evaluate(model_arch, X_val, y_val)
        metrics_arch["training_time"] = training_time
        log_metrics(metrics_arch)
        results_architectures[arch_name] = metrics_arch

        print(f"Arch {arch_name:15s}: accuracy={metrics_arch['accuracy']:.4f}, tempo={training_time:.2f}s")

df_arch = pd.DataFrame(results_architectures).T
print("\nComparação de Arquiteturas:")
print(df_arch[["accuracy", "f1_score", "training_time"]].round(4).to_string())

compare_models(
    list(results_architectures.keys()),
    [m["accuracy"] for m in results_architectures.values()],
    "Accuracy por Arquitetura"
)

### Respostas — Questão 6

**1. Redes maiores sempre melhoraram os resultados?**

Não. Redes maiores possuem maior capacidade representacional, mas precisam de **mais iterações e mais dados** para ajustar todos os seus parâmetros. Com apenas 20 iterações, arquiteturas menores podem competir com — ou superar — arquiteturas maiores, que ainda não tiveram tempo suficiente para convergir.

**2. Qual arquitetura apresentou melhor tradeoff?**

A arquitetura **(128, 64)** geralmente apresenta o melhor tradeoff: é suficientemente expressiva para capturar padrões complexos do CIFAR-100, mas não exige um número excessivo de iterações para convergir nem impõe custo computacional proibitivo.

**3. Houve sinais de overfitting?**

Com apenas 20 iterações, o overfitting clássico (accuracy de treino muito acima da de validação) é difícil de observar diretamente. No entanto, arquiteturas maiores como `(256, 128)` tendem a exibir maior diferença entre as métricas de treino e validação, sugerindo que sua capacidade ainda não foi totalmente aproveitada — ou que há início de sobreajuste.

**4. Qual arquitetura apresentou maior custo computacional?**

A arquitetura **(256, 128)** apresenta o maior custo, pois possui mais parâmetros e realiza mais operações matriciais por mini-batch. O número de multiplicações por camada cresce proporcionalmente ao produto entre o tamanho da camada anterior e o da camada seguinte — com 256 e 128 neurônios, esse produto é substancialmente maior que em arquiteturas menores.

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
learning_rates = [0.1, 0.01, 0.001]
results_lr = {}

for lr in learning_rates:
    lr_name = str(lr)
    with start_run(f"lr_{lr_name}"):
        log_params({
            "activation": "relu",
            "hidden_layers": str((128, 64)),
            "learning_rate": lr,
            "max_iter": 20,
            "batch_size": 512
        })

        start_time = time.time()
        model_lr = MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            learning_rate_init=lr,
            max_iter=20,
            batch_size=512,
            random_state=SEED
        ).fit(X_train, y_train)
        training_time = time.time() - start_time

        metrics_lr = evaluate(model_lr, X_val, y_val)
        metrics_lr["training_time"] = training_time
        log_metrics(metrics_lr)
        results_lr[lr_name] = metrics_lr

        print(f"LR={lr_name:5s}: accuracy={metrics_lr['accuracy']:.4f}, f1={metrics_lr['f1_score']:.4f}, tempo={training_time:.2f}s")

df_lr = pd.DataFrame(results_lr).T
print("\nComparação de Learning Rates:")
print(df_lr[["accuracy", "f1_score", "training_time"]].round(4).to_string())

compare_models(
    list(results_lr.keys()),
    [m["accuracy"] for m in results_lr.values()],
    "Accuracy por Learning Rate"
)

### Respostas — Questão 7

**1. Qual learning rate apresentou melhor desempenho?**

O **lr = 0.01** geralmente apresenta o melhor desempenho: é grande o suficiente para convergir dentro de 20 iterações e pequeno o suficiente para não causar oscilações na função de perda.

**2. Qual apresentou maior instabilidade?**

O **lr = 0.1** apresenta maior instabilidade. Passos tão grandes no espaço de parâmetros podem levar o otimizador a "saltar" sobre mínimos da função de perda, gerando oscilações na loss e, em casos extremos, divergência do treinamento.

**3. O que acontece quando o learning rate é muito alto?**

O otimizador realiza atualizações de pesos excessivamente grandes, podendo:

- **ultrapassar o mínimo** da função de perda e causar oscilação permanente em torno dele;
- **aumentar a loss** em vez de diminuí-la (divergência);
- impedir que o modelo encontre qualquer solução estável, independentemente do número de iterações.

**4. O que acontece quando o learning rate é muito baixo?**

A convergência torna-se extremamente lenta: cada atualização modifica os pesos por uma quantidade ínfima. Com um número fixo de iterações (`max_iter=20`), o modelo pode encerrar o treinamento muito distante de um bom mínimo — resultando em baixo desempenho mesmo com uma arquitetura e ativação adequadas.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

## Discussão — Questão 8

### Comportamento da Loss

A loss da MLP decresce mais rapidamente nas primeiras iterações e estabiliza progressivamente. Com `max_iter=20`, a maioria dos modelos não converge completamente — o que é suficiente para fins comparativos, mas evidencia a limitação de treinar uma MLP com tantas features e tão poucas épocas. A taxa de decrescimento da loss é determinada principalmente pelo learning rate e pela arquitetura.


### Impacto da Arquitetura

Arquiteturas maiores possuem maior capacidade representacional, mas exigem mais iterações para convergir e impõem maior custo computacional. Com apenas 20 iterações, a arquitetura `(128, 64)` oferece o melhor tradeoff. Redes maiores como `(256, 128)` tendem a subaprender no regime de poucas épocas.

### Impacto das Funções de Ativação

| Ativação | Característica |
|---|---|
| **ReLU** | Melhor convergência; não satura em valores positivos |
| **tanh** | Estabilidade similar à ReLU; gradientes mais suaves |
| **logistic** | Convergência mais lenta; saturação em ambos os extremos |

### Comportamento do Treinamento

O SGD com mini-batches converge de forma ruidosa mas eficiente. Com `batch_size=512`, o número de passos por época é reduzido (≈ 78 passos para 40.000 amostras de treino), o que acelera o treinamento em tempo de relógio, mas introduz mais ruído nos gradientes comparado a batch maior.

### Limitações da MLP para Imagens

A MLP trata cada pixel como uma feature independente, desconsiderando:

- **Estrutura espacial local** — a relação de vizinhança entre pixels adjacentes é ignorada;
- **Invariância geométrica** — translação, rotação ou pequenas deformações da imagem geram representações completamente diferentes;
- **Hierarquia de features** — CNNs aprendem progressivamente bordas → texturas → partes → objetos; a MLP não tem essa indução.

Consequentemente, o número de parâmetros cresce muito mais rápido que a capacidade efetiva de generalização.

### Relação entre Backpropagation e Aprendizado

O backpropagation aplica a **regra da cadeia** para calcular o gradiente da função de perda em relação a cada peso da rede, propagando o sinal de erro da camada de saída para as camadas anteriores. O SGD usa esses gradientes para atualizar os pesos iterativamente na direção que minimiza o erro. Sem backpropagation, treinar redes com múltiplas camadas ocultas seria computacionalmente inviável.

---

### Respostas diretas

**1. Qual configuração apresentou melhor resultado final?**

`ReLU + (128, 64) + lr=0.01` — combina convergência rápida (ReLU), capacidade suficiente para o CIFAR-100 (128, 64) e estabilidade de treinamento (lr=0.01).

**2. Quais foram as principais dificuldades observadas?**

O CIFAR-100 possui 100 classes com apenas ~600 amostras por classe no treino — pouco dado por classe para a alta dimensionalidade (3.072 features) tratada sem exploração espacial. Além disso, 20 iterações são insuficientes para convergência plena, limitando o potencial de todas as arquiteturas.

**3. Por que MLPs possuem limitações para imagens?**

Porque não exploram a estrutura espacial, não têm invariância a transformações geométricas e crescem quadraticamente em parâmetros com o tamanho da entrada — tudo que as CNNs resolvem com compartilhamento de pesos e pooling.

**4. Como o backpropagation contribui para o aprendizado da rede?**

Ele calcula, de forma eficiente, o gradiente da perda em relação a cada peso usando a regra da cadeia — tornando viável o treinamento de redes com milhões de parâmetros. Sem backpropagation, seria necessário estimar cada gradiente por diferenças finitas, o que é impraticável em alta dimensionalidade.